In [0]:
import pandas as pd
import numpy as np

# Only cities A and B
CITIES = ["toronto", "vancouver"]
FNAME = "listings.csv.gz"

city_summary = []
dfs = {}

for city in CITIES:
    path = f"/Workspace/Shared/canada-airbnb/data/{city}/{FNAME}"
    try:
        df = pd.read_csv(path, compression="gzip", low_memory=False)
        dfs[city] = df
        
        # Check the existence of snapshot_date or change for calendar_last_scraped / last_scraped
        snapshot_col = 'last_scraped' if 'last_scraped' in df.columns else 'calendar_last_scraped'
        snapshot_date = df[snapshot_col].dropna().iloc[0] if snapshot_col in df.columns else "Unknown"
        
        city_summary.append({
            "City": city,
            "Rows": len(df),
            "Columns": len(df.columns),
            "Snapshot Date": snapshot_date
        })
    except Exception as e:
        print(f"Error loading {city}: {e}")

summary_df = pd.DataFrame(city_summary)
display(summary_df)

if "toronto" in dfs:
    t_df = dfs["toronto"]
    info_df = pd.DataFrame({
        "column": t_df.columns,
        "dtype": t_df.dtypes.values,
        "null_count": t_df.isnull().sum().values,
        "null_pct": (t_df.isnull().mean() * 100).round(2).values,
        "sample_val": [t_df[col].dropna().iloc[0] if not t_df[col].dropna().empty else None for col in t_df.columns]
    })
    print("\n--- Sample of Toronto ---")
    display(info_df.head(30))

# 01. Exploratory Data Analysis & Leakage Review
**Cities in Analysis:** Toronto, Vancouver  
*(Note: Montreal is deliberately excluded from analysis until model readiness verification)*

## Executive Summary

### What we already know about the data:
1. **Volume and Structure:** Toronto's sample size (22,050 rows) is nearly 3.6 times larger than Vancouver's (6,131 rows). Both datasets share an identical 90-column schema and consistent snapshot dates (September 2026).
2. **Systemic Missing Values (100% null):** In the current snapshot, several systemic columns are completely empty (`neighborhood_overview`, `host_since`, `host_response_time`, `host_response_rate`, `host_acceptance_rate`, `host_thumbnail_url`, `host_neighbourhood`, `host_total_listings_count`). They carry no informative value and should be dropped.
3. **Host Tenure Parameters:** The dataset contains pre-computed numerical host and user tenure features (`hosts_time_as_user_years`, `hosts_time_as_host_years`), replacing the missing `host_since` column.

### What remains unknown / needs pipeline verification:
1. **Price Data Types:** The `price` column contains string formatting (e.g., `$150.00`) with thousands separators, requiring cleaning to `float` data types.
2. **Anomalous Price Outliers:** Upper bound thresholds for price filtering need to be established (`price == 0` and extreme high prices from $2,500+ per night) to prevent distortion during model training.
3. **Geographic Density:** Further spatial missingness checks are required for geographical coordinates (`latitude`, `longitude`) and neighborhood names (`neighbourhood_cleansed`).

## 1. Dataset Description
- **Grain:** 1 row = 1 unique short-term rental listing per snapshot date.

In [0]:
import pandas as pd
import numpy as np

# Create dataset description table
dataset_desc = pd.DataFrame([
    {
        "City": "Toronto",
        "Snapshot Date": "2026-09-18",
        "Record Count": 22050,
        "Total Columns": 90,
        "Grain": "1 row per listing"
    },
    {
        "City": "Vancouver",
        "Snapshot Date": "2026-09-17",
        "Record Count": 6131,
        "Total Columns": 90,
        "Grain": "1 row per listing"
    }
])

display(dataset_desc)

## 2. Column Profiling, Null Patterns & Outliers

### Missing Value Patterns:
* **Critical Missingness (100% missing):** Columns such as `host_response_time`, `host_response_rate`, `host_acceptance_rate`, and `neighborhood_overview` have null values for 100% of records in all cities. They are dropped in the Silver layer.
* **Categorical Missingness:** `host_location` has 29.44% missing values in Toronto. To be imputed with `'Unknown'`.
* **Numerical Missingness in Property Features:** Columns like `bedrooms`, `beds`, and `bathrooms` have a low missing rate (< 2%). Imputed using the median grouped by `room_type`.

### Outliers Classification & Handling:
1. **Price (`price`):**
   * **Low Outliers:** `price <= 0` records are classified as data entry errors and removed from the training set.
   * **High Outliers:** Records above the 99th percentile (or $> \$2,500$ CAD) are filtered out for the ML model, but retained in a dedicated Silver-layer segment for BI analytics.
2. **Availability (`availability_365`):** Values are strictly bounded within $[0, 365]$. Properties with `availability_365 = 0` are retained but flagged as inactive.

In [0]:
# Function for cleaning price and inspecting price distributions
def clean_price_series(series):
    if series.dtype == 'object':
        return series.str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)
    return series

for city in ["toronto", "vancouver"]:
    df = dfs[city].copy()
    df['price_clean'] = clean_price_series(df['price'])
    
    p99 = df['price_clean'].quantile(0.99)
    p50 = df['price_clean'].median()
    p_min = df['price_clean'].min()
    p_max = df['price_clean'].max()
    
    print(f"=== STATS FOR {city.upper()} ===")
    print(f"Median Price: ${p50:.2f} CAD")
    print(f"99th Percentile Price: ${p99:.2f} CAD")
    print(f"Min Price: ${p_min:.2f} | Max Price: ${p_max:.2f}")
    print(f"Count of zero/negative prices: {(df['price_clean'] <= 0).sum()}")
    print("-" * 40)

In [0]:
%pip install seaborn

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Calculate missing value percentages across non-100% missing columns
missing_records = []
for city, df in dfs.items():
    null_pct = (df.isnull().mean() * 100).round(2)
    # Filter columns with missing values between 0.1% and 90%
    partial_nulls = null_pct[(null_pct > 0.1) & (null_pct < 90)]
    for col, pct in partial_nulls.items():
        missing_records.append({"City": city.capitalize(), "Column": col, "Missing_Pct": pct})

missing_df = pd.DataFrame(missing_records)

if not missing_df.empty:
    plt.figure(figsize=(12, 6))
    sns.barplot(data=missing_df, x="Missing_Pct", y="Column", hue="City", palette="Set2")
    plt.title("Partial Missing Value Rate (%) by Column & City", fontsize=14, fontweight="bold")
    plt.xlabel("Missing Percentage (%)")
    plt.ylabel("Column Feature")
    plt.tight_layout()
    plt.show()

In [0]:
# Price and Log-Price distribution visual check
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for i, (city, df) in enumerate(dfs.items()):
    clean_p = clean_price_series(df['price'])
    # Filter valid range for training visualization
    valid_p = clean_p[(clean_p > 0) & (clean_p <= 1000)]
    
    # Raw Price Distribution (< $1000)
    sns.histplot(valid_p, bins=40, kde=True, ax=axes[0, i], color="skyblue" if i==0 else "salmon")
    axes[0, i].set_title(f"Raw Price Distribution (< $1000 CAD) - {city.capitalize()}", fontweight="bold")
    axes[0, i].set_xlabel("Price (CAD)")
    axes[0, i].set_ylabel("Listing Count")
    
    # Log-Transformed Price Distribution
    sns.histplot(np.log1p(valid_p), bins=40, kde=True, ax=axes[1, i], color="steelblue" if i==0 else "coral")
    axes[1, i].set_title(f"Log(Price + 1) Distribution - {city.capitalize()}", fontweight="bold")
    axes[1, i].set_xlabel("Log(Price)")
    axes[1, i].set_ylabel("Listing Count")

plt.tight_layout()
plt.show()

In [0]:
# Availability 365 Distribution
plt.figure(figsize=(12, 5))
avail_data = []

for city, df in dfs.items():
    if 'availability_365' in df.columns:
        temp = df[['availability_365']].copy()
        temp['City'] = city.capitalize()
        avail_data.append(temp)

if avail_data:
    combined_avail = pd.concat(avail_data)
    sns.histplot(
        data=combined_avail, 
        x="availability_365", 
        hue="City", 
        element="step", 
        stat="density", 
        common_norm=False, 
        bins=36,
        palette="Set1"
    )
    plt.title("Distribution of Yearly Availability (availability_365)", fontsize=14, fontweight="bold")
    plt.xlabel("Days Available per Year")
    plt.ylabel("Density")
    plt.tight_layout()
    plt.show()

In [0]:
# Price distribution by Room Type (Outlier truncated at $1000 for display)
price_room_list = []
for city, df in dfs.items():
    temp = df[['room_type', 'price']].copy()
    temp['price_clean'] = clean_price_series(temp['price'])
    temp['City'] = city.capitalize()
    price_room_list.append(temp[(temp['price_clean'] > 0) & (temp['price_clean'] <= 1000)])

combined_pr = pd.concat(price_room_list)

plt.figure(figsize=(12, 6))
sns.boxplot(data=combined_pr, x="room_type", y="price_clean", hue="City", palette="Blues")
plt.title("Price Distribution by Room Type (Listings <= $1000 CAD)", fontsize=14, fontweight="bold")
plt.xlabel("Room Type")
plt.ylabel("Price per Night (CAD)")
plt.tight_layout()
plt.show()

## 3. Data Leakage Review

The data leakage audit identifies variables that **MUST NOT** be used to predict price due to direct mathematical dependency or post-event logging.

### Leaked / Invalid Features (Prohibited for Modeling):

| Column | Risk Category | Why it's a Leak (Reason) |
| :--- | :--- | :--- |
| `weekly_price` / `monthly_price` | Direct Derivative | Calculated directly based on the daily `price`. |
| `minimum_nights_avg_ntm` / `maximum_nights_avg_ntm` | Derived Metrics | Dynamic metrics computed over future calendar booking periods. |
| `number_of_reviews_*` / `review_scores_*` | Historical / Post-event Leak | Reviews and ratings accumulate *after* a guest stays at the set price. New listings lack these features. |
| `estimated_revenue` / `occupancy_rate` | Financial Derivative Metrics | Calculated as `price * estimated_nights_booked`. Including them introduces target leakage. |
| `license` | Administrative Leak | Not an intrinsic property characteristic; licensing status varies by city policy and lacks values in new cities. |
| `calendar_updated` | Future State | Displays the system timestamp of calendar updates following price adjustments. |

> **Rule for ML Model:** The model must **ONLY** utilize a priori physical listing characteristics (number of rooms, bedrooms, guests, property type, neighborhood, geographic coordinates) and host tenure at the time of creation.

## 4. Cleaning & Transformations (ETL Rules for Person 1 Pipeline)

To be implemented in `02_etl_pipeline.ipynb`, the following mandatory transformation rules apply when moving data from Bronze to Silver:

1. **Price Data Typing:** Convert `price` from string (`"$120.00"`) to `float` (`120.00`)[cite: 3].
2. **Anomaly Filtering:**
   * Drop rows where `price <= 0` or `price IS NULL`[cite: 3].
   * Truncate extreme upper tail: filter `price > 2500` for the training dataset[cite: 3].
3. **Categorical Text Cleaning:**
   * Normalize `room_type` and `property_type` (lowercase, strip extra whitespace).
4. **Missing Value Imputation:**
   * `bedrooms`, `beds`, `bathrooms` $\rightarrow$ impute missing values using median grouped by `room_type`.
   * `host_is_superhost` $\rightarrow$ impute missing values with `'f'`.
5. **Dropping Unusable Columns:** Remove 100% empty columns (`neighborhood_overview`, `host_response_time`, etc.) and media URLs (`picture_url`, `host_url`).